# 04. Batting Performance Analysis
**Project:** Sports Performance Analytics Using IPL Data  
**Domain:** Sports Analytics & Business Intelligence  

---

### Notebook Objective
1. Compute career batting aggregates: Runs, Balls Faced, Fours, Sixes, Dismissals, and Highest Scores.
2. Calculate Batting Average and Batting Strike Rate adhering strictly to legal deliveries and not-out adjustments.
3. Identify the Top 10 All-Time Run Scorers in IPL history.
4. Analyze boundary run contributions (4s and 6s percentage).
5. Generate quadrant scatter plot: Batting Average vs Strike Rate.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

clean_dir = os.path.join(PROJECT_DIR, 'Cleaned_Data')
viz_dir = os.path.join(PROJECT_DIR, 'Visualizations')

deliveries = pd.read_csv(os.path.join(clean_dir, 'deliveries.csv'), low_memory=False)


### 1. Feature Engineering Batting Aggregates


In [2]:
df_bat = deliveries[~deliveries['is_super_over']].copy()

match_bat = df_bat.groupby(['match_id', 'striker']).agg(
    runs=('runs_off_bat', 'sum'),
    balls_faced=('is_ball_faced', 'sum'),
    fours=('is_four', 'sum'),
    sixes=('is_six', 'sum')
).reset_index()

dismissals = df_bat[df_bat['player_dismissed'].notna()].groupby(['match_id', 'player_dismissed']).size().reset_index(name='dismissed')
match_bat = match_bat.merge(dismissals, left_on=['match_id', 'striker'], right_on=['match_id', 'player_dismissed'], how='left')
match_bat['is_dismissed'] = match_bat['dismissed'].fillna(0) > 0

batting_stats = match_bat.groupby('striker').agg(
    matches=('match_id', 'nunique'),
    innings=('match_id', 'count'),
    runs=('runs', 'sum'),
    balls_faced=('balls_faced', 'sum'),
    dismissals=('is_dismissed', 'sum'),
    fours=('fours', 'sum'),
    sixes=('sixes', 'sum'),
    highest_score=('runs', 'max')
).reset_index().rename(columns={'striker': 'player'})

batting_stats['not_outs'] = (batting_stats['innings'] - batting_stats['dismissals']).clip(lower=0)
batting_stats['batting_average'] = np.where(
    batting_stats['dismissals'] > 0,
    (batting_stats['runs'] / batting_stats['dismissals']).round(2),
    batting_stats['runs'].astype(float)
)
batting_stats['strike_rate'] = np.where(
    batting_stats['balls_faced'] > 0,
    ((batting_stats['runs'] / batting_stats['balls_faced']) * 100).round(2),
    0.0
)
batting_stats['boundary_runs'] = (batting_stats['fours'] * 4) + (batting_stats['sixes'] * 6)
batting_stats['boundary_percentage'] = np.where(
    batting_stats['runs'] > 0,
    ((batting_stats['boundary_runs'] / batting_stats['runs']) * 100).round(2),
    0.0
)
batting_stats = batting_stats.sort_values(by='runs', ascending=False).reset_index(drop=True)


### 2. Top 10 Run Scorers Leaderboard


In [3]:
top10 = batting_stats.head(10)[['player', 'innings', 'runs', 'highest_score', 'batting_average', 'strike_rate', 'fours', 'sixes', 'boundary_percentage']]
top10


,player,innings,runs,highest_score,batting_average,strike_rate,fours,sixes,boundary_percentage
0,V Kohli,275,9336,113,40.42,134.80,845,317,56.58
1,RG Sharma,275,7329,109,29.91,132.92,661,324,62.60
2,S Dhawan,221,6769,106,35.07,127.14,768,153,58.95
3,DA Warner,184,6565,126,40.52,139.77,663,236,61.96
4,KL Rahul,149,5815,152,46.52,139.15,508,239,59.60
5,SK Raina,200,5528,100,32.52,136.73,506,203,58.65
6,MS Dhoni,241,5439,84,38.30,137.45,375,264,56.70
7,AM Rahane,197,5365,105,30.31,125.61,540,141,56.03
8,SV Samson,185,5180,119,31.78,141.14,432,243,61.51
9,AB de Villiers,170,5162,133,39.71,151.69,413,251,61.18


In [4]:
plt.figure(figsize=(10, 5))
top10_plot = top10.sort_values(by='runs', ascending=True)
bars = plt.barh(top10_plot['player'], top10_plot['runs'], color='#2ca02c')
plt.title("Top 10 Run Scorers in IPL History", fontsize=13, fontweight='bold')
plt.xlabel("Total Career Runs")
plt.ylabel("Player")
for b in bars:
    w = b.get_width()
    plt.annotate(f"{int(w):,}", xy=(w, b.get_y() + b.get_height()/2), xytext=(5, 0), textcoords="offset points", va='center')
plt.tight_layout()
plt.savefig(os.path.join(viz_dir, '03_top_10_run_scorers.png'), dpi=200)
plt.show()


### 3. Highest Individual Innings in IPL History


In [5]:
top_scores = match_bat.sort_values(by='runs', ascending=False).head(10)[['striker', 'runs', 'balls_faced', 'fours', 'sixes']]
top_scores.rename(columns={'striker': 'player'}).reset_index(drop=True)


### 4. Batting Average vs Strike Rate Quadrant Analysis
Min 25 innings qualification threshold.


In [6]:
plt.figure(figsize=(9, 6))
qual = batting_stats[batting_stats['innings'] >= 25].head(40)
scatter = plt.scatter(qual['batting_average'], qual['strike_rate'], s=qual['runs']/20, c=qual['boundary_percentage'], cmap='viridis', alpha=0.85)
plt.axvline(30, color='red', linestyle='--', label='Average Benchmark (30.0)')
plt.axhline(130, color='blue', linestyle='--', label='Strike Rate Benchmark (130.0)')
plt.title("Batting Average vs Strike Rate (Min 25 Innings)", fontsize=13, fontweight='bold')
plt.xlabel("Batting Average")
plt.ylabel("Strike Rate")
plt.colorbar(scatter, label='Boundary %')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(viz_dir, '04_batting_strike_rate_vs_average.png'), dpi=200)
plt.show()


### 5. Batting Insights
- Virat Kohli, Shikhar Dhawan, and David Warner form the elite top-tier run scorers surpassing 6,500+ career runs.
- Chris Gayle's historic 175* off 66 balls remains the highest individual T20 innings of all time.
- Elite T20 batters occupy the top-right quadrant (Batting Average > 30 and Strike Rate > 130), with AB de Villiers and David Warner displaying exceptional balance of longevity and destructive scoring.
